# 01 - Exploratory Data Analysis (EDA)
## Building Energy Efficiency: Heating & Cooling Load Prediction
**DEPI Graduation Project - Team CloudXAi (NHA-5-165)**

This notebook investigates the statistical properties, distributions, correlations, and multicollinearity of the 8 architectural features and 2 energy load targets.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.energy_efficiency.config import RAW_DATA_PATH, FEATURES, TARGETS, REPORTS_DIR
from src.energy_efficiency.data_loader import load_raw_data

sns.set_theme(style="whitegrid")
print("Environment and configuration initialized.")


### 1. Data Ingestion & Quality Checks


In [ ]:
df = load_raw_data()
print("Dataset Shape:", df.shape)
display(df.head())


In [ ]:
print("=== Missing Values ===")
print(df.isnull().sum())

print("\n=== Duplicate Records ===")
print("Duplicates found:", df.duplicated().sum())

print("\n=== Summary Statistics ===")
display(df.describe().round(2))


### 2. Feature Distributions & Boxplots


In [ ]:
all_cols = FEATURES + TARGETS
fig, axes = plt.subplots(3, 4, figsize=(14, 8))
for idx, col in enumerate(all_cols):
    r, c = divmod(idx, 4)
    sns.histplot(df[col], kde=True, ax=axes[r, c], color="steelblue")
    axes[r, c].set_title(f"Distribution: {col}", fontsize=9)
for idx in range(len(all_cols), 12):
    r, c = divmod(idx, 4)
    fig.delaxes(axes[r, c])
plt.tight_layout()
plt.show()


### 3. Correlation Analysis & Multicollinearity


In [ ]:
plt.figure(figsize=(9, 7))
corr = df.corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Correlation Matrix (Features & Targets)")
plt.show()


In [ ]:
# Identify pairs with correlation > 0.80
upper_tri = corr.abs().where(np.triu(np.ones(corr.shape), k=1).astype(bool))
high_corr = upper_tri.stack().reset_index()
high_corr.columns = ["Feature 1", "Feature 2", "Abs Correlation"]
high_corr = high_corr[high_corr["Abs Correlation"] > 0.8].sort_values("Abs Correlation", ascending=False)
display(high_corr)


### 4. Features vs Targets & Relationship Insights


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.scatterplot(data=df, x="Relative_Compactness", y="Heating_Load", hue="Overall_Height", palette="viridis", ax=axes[0])
axes[0].set_title("Relative Compactness vs Heating Load")

sns.scatterplot(data=df, x="Heating_Load", y="Cooling_Load", hue="Overall_Height", palette="magma", ax=axes[1])
axes[1].set_title("Heating Load vs Cooling Load")
plt.tight_layout()
plt.show()


### 5. Summary & Key Takeaways
1. **Multicollinearity:** Relative Compactness and Surface Area have a near-perfect inverse correlation ($r = -0.99$).
2. **Key Drivers:** Overall Height and Roof Area are the primary determinants of building thermal load.
3. **Target Correlation:** Heating Load and Cooling Load are strongly co-dependent ($r = 0.98$).
